## Dimensão de Calendário

Tarefa:
- Construa uma dimensão de datas utilizando sql
- Cruze a dimensão de datas com a tabela de vendas para análise (não esqueça de considerar os dias sem vendas).

In [1]:
import pandas as pd
import duckdb

df_vendas = pd.read_csv('../datasets/vendas_2023_2024.csv')

In [2]:
query = """
-- Dimensão de datas: todos os dias entre a menor e maior data de venda
WITH calendario AS (
    SELECT UNNEST(
        generate_series(
            MIN(CASE
                WHEN regexp_matches(sale_date, '^\d{4}-\d{2}-\d{2}$') THEN strptime(sale_date, '%Y-%m-%d')
                WHEN regexp_matches(sale_date, '^\d{2}-\d{2}-\d{4}$') THEN strptime(sale_date, '%d-%m-%Y')
            END)::DATE,
            MAX(CASE
                WHEN regexp_matches(sale_date, '^\d{4}-\d{2}-\d{2}$') THEN strptime(sale_date, '%Y-%m-%d')
                WHEN regexp_matches(sale_date, '^\d{2}-\d{2}-\d{4}$') THEN strptime(sale_date, '%d-%m-%Y')
            END)::DATE,
            INTERVAL '1 day'
        )
    ) AS data_ref
    FROM df_vendas
),

-- Vendas diárias (soma por dia)
vendas_diarias AS (
    SELECT
        CASE
            WHEN regexp_matches(sale_date, '^\d{4}-\d{2}-\d{2}$') THEN strptime(sale_date, '%Y-%m-%d')
            WHEN regexp_matches(sale_date, '^\d{2}-\d{2}-\d{4}$') THEN strptime(sale_date, '%d-%m-%Y')
        END::DATE AS data_venda,
        SUM(total) AS total_dia
    FROM df_vendas
    GROUP BY data_venda
),

-- Cruzamento: dias sem venda recebem 0
calendario_vendas AS (
    SELECT
        c.data_ref,
        DAYOFWEEK(c.data_ref)                           AS num_dia,
        CASE DAYOFWEEK(c.data_ref)
            WHEN 0 THEN 'Domingo'
            WHEN 1 THEN 'Segunda-feira'
            WHEN 2 THEN 'Terça-feira'
            WHEN 3 THEN 'Quarta-feira'
            WHEN 4 THEN 'Quinta-feira'
            WHEN 5 THEN 'Sexta-feira'
            WHEN 6 THEN 'Sábado'
        END                                             AS dia_semana,
        COALESCE(v.total_dia, 0)                        AS valor_venda
    FROM calendario c
    LEFT JOIN vendas_diarias v ON v.data_venda = c.data_ref
)

-- Média por dia da semana
SELECT
    dia_semana,
    COUNT(*)                                AS total_dias,
    SUM(valor_venda)                        AS receita_total,
    ROUND(AVG(valor_venda), 2)              AS media_vendas
FROM calendario_vendas
GROUP BY dia_semana, num_dia
ORDER BY num_dia ASC
"""

df_calendario = duckdb.query(query).df()
df_calendario

<>:7: SyntaxWarning: invalid escape sequence '\d'
<>:7: SyntaxWarning: invalid escape sequence '\d'
C:\Users\livma\AppData\Local\Temp\ipykernel_5796\4044969507.py:7: SyntaxWarning: invalid escape sequence '\d'
  WHEN regexp_matches(sale_date, '^\d{4}-\d{2}-\d{2}$') THEN strptime(sale_date, '%Y-%m-%d')


,dia_semana,total_dias,receita_total,media_vendas
0,Domingo,105,3.485479e+08,3319503.57
1,Segunda-feira,105,3.638395e+08,3465137.71
2,Terça-feira,105,3.808398e+08,3627045.76
3,Quarta-feira,104,3.676676e+08,3535265.63
4,Quinta-feira,104,3.771282e+08,3626232.44
5,Sexta-feira,104,3.863604e+08,3715003.41
6,Sábado,104,3.858962e+08,3710540.55


In [3]:
df_calendario.sort_values('media_vendas').head(1)[['dia_semana', 'media_vendas']]

,dia_semana,media_vendas
0,Domingo,3319503.57


**METODOLOGIA — Questão 6**

1. POR QUE USAR UMA TABELA DE DATAS?
  - Agrupar diretamente a tabela de vendas ignora os dias em que a loja
   abriu mas não registrou nenhuma venda — esses dias simplesmente não
   existem na tabela. A dimensão de calendário garante que todos os dias
   do período estejam representados, e o LEFT JOIN preenche com zero os
   dias sem movimento. Sem isso, a média fica inflada porque o denominador
   considera apenas dias com venda, não todos os dias em que a loja operou.

2. O QUE ACONTECE SEM OS DIAS ZERADOS?
  - Se um dia da semana tiver muitos dias sem venda, sua média calculada
   diretamente da tabela de vendas será artificialmente alta — porque só
   os dias com movimento entram no cálculo. Um dia que vendeu R$ 5.000
   em 10 das 20 terças-feiras teria média de R$ 5.000 pelo método do
   estagiário, mas média real de R$ 2.500 quando os 10 dias zerados
   são incluídos. A dimensão de datas corrige exatamente esse viés.